<a href="https://colab.research.google.com/github/JeysonCarmona/PPMI_INVESTIGATION/blob/main/notebook2_exomas_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Exome Analysis (PPMI)

**Objective:** Iterate through the 78 `.tar.gz` exome files **without fully decompressing them**, read only their internal structure (list of tar members), extract the patient identifier from each `.vcf` file name, validate the naming pattern, detect duplicates and corrupted files, and generate `exomas_index.csv`.

In [ ]:
!apt-get install -y unrar -qq
!pip install rarfile -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Investigación_Parkinson"
EXOMAS_DIR = BASE_DIR + "/Exomas - Parkinson"
RESULTADOS_DIR = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados"

import os
os.makedirs(RESULTADOS_DIR, exist_ok=True)

assert os.path.isdir(EXOMAS_DIR), f"No existe la carpeta de exomas: {EXOMAS_DIR}"
print("Carpeta de exomas OK:", EXOMAS_DIR)


Carpeta de exomas OK: /content/drive/MyDrive/Investigación_Parkinson/Exomas - Parkinson


In [ ]:
import tarfile
import re
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


## 1. List of available TAR files

In [ ]:
tar_files = sorted([
    f for f in os.listdir(EXOMAS_DIR)
    if f.endswith(".tar.gz")
])

print("Archivos .tar.gz encontrados:", len(tar_files))
for f in tar_files[:5]:
    print(" -", f)
if len(tar_files) > 5:
    print("  ...")


Archivos .tar.gz encontrados: 78
 - ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz
 - ppmi_wes_645_indiv_vcf_set_02_of_78.tar.gz
 - ppmi_wes_645_indiv_vcf_set_03_of_78.tar.gz
 - ppmi_wes_645_indiv_vcf_set_04_of_78.tar.gz
 - ppmi_wes_645_indiv_vcf_set_05_of_78.tar.gz
  ...


In [ ]:
PATRON_TAR = re.compile(r"ppmi_wes_\d+_indiv_vcf_set_(\d+)_of_(\d+)\.tar\.gz")

tars_no_conformes = [f for f in tar_files if not PATRON_TAR.match(f)]
print("TAR que NO siguen el patrón esperado:", len(tars_no_conformes))
if tars_no_conformes:
    print(tars_no_conformes)


TAR que NO siguen el patrón esperado: 0


## 2. Reading the internal structure of each TAR (without extracting)

`tarfile.open(..., 'r:gz')` allows reading the list of members (`getmembers()`/`getnames()`) without decompressing the content to disk. Only the tar index (headers) is read, which is fast even for large files.

In [ ]:
PATRON_VCF = re.compile(r"PPMI_SI_(\d+)\.raw\.vcf$")

def leer_estructura_tar(ruta_tar):
    """Lee los nombres de los miembros de un tar.gz sin extraer contenido.
    Devuelve una lista de dicts: nombre, patno, tamaño, es_vcf_valido.
    Si el tar está corrupto, devuelve None y el motivo del error.
    """
    registros = []
    try:
        with tarfile.open(ruta_tar, "r:gz") as tar:
            for member in tar.getmembers():
                nombre = os.path.basename(member.name)
                match = PATRON_VCF.match(nombre)
                registros.append({
                    "tar_file": os.path.basename(ruta_tar),
                    "nombre_archivo": nombre,
                    "ruta_interna": member.name,
                    "tamano_bytes": member.size,
                    "patno": int(match.group(1)) if match else None,
                    "formato_valido": match is not None,
                    "es_archivo": member.isfile(),
                })
        return registros, None
    except Exception as e:
        return None, str(e)


In [ ]:
todos_los_registros = []
tars_corruptos = []

for i, tar_name in enumerate(tar_files, start=1):
    ruta = os.path.join(EXOMAS_DIR, tar_name)
    registros, error = leer_estructura_tar(ruta)
    if error is not None:
        print(f"[{i}/{len(tar_files)}] ERROR leyendo {tar_name}: {error}")
        tars_corruptos.append({"tar_file": tar_name, "error": error})
        continue
    todos_los_registros.extend(registros)
    if i % 10 == 0 or i == len(tar_files):
        print(f"[{i}/{len(tar_files)}] procesados. Registros acumulados: {len(todos_los_registros)}")

print("\nLectura de estructura completada.")
print("TAR procesados correctamente:", len(tar_files) - len(tars_corruptos))
print("TAR corruptos / con error:", len(tars_corruptos))


[10/78] procesados. Registros acumulados: 186
[20/78] procesados. Registros acumulados: 376
[30/78] procesados. Registros acumulados: 566
[40/78] procesados. Registros acumulados: 756
[50/78] procesados. Registros acumulados: 946
[60/78] procesados. Registros acumulados: 1126
[70/78] procesados. Registros acumulados: 1280
[78/78] procesados. Registros acumulados: 1358

Lectura de estructura completada.
TAR procesados correctamente: 78
TAR corruptos / con error: 0


In [ ]:
df_exomas_raw = pd.DataFrame(todos_los_registros)
# Solo nos interesan los archivos .vcf reales, no directorios internos
df_exomas = df_exomas_raw[
    (df_exomas_raw["es_archivo"]) & (df_exomas_raw["nombre_archivo"].str.endswith(".vcf"))
].copy()

print("Total de entradas de archivo (todas):", len(df_exomas_raw))
print("Total de archivos .vcf:", len(df_exomas))
df_exomas.head()


Total de entradas de archivo (todas): 1358
Total de archivos .vcf: 645


,tar_file,nombre_archivo,ruta_interna,tamano_bytes,patno,formato_valido,es_archivo
1,ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz,PPMI_SI_3050.raw.vcf,ppmi_wes_645_indiv_vcf_set_01_of_78/PPMI_SI_30...,4735792499,3050.0,True,True
2,ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz,PPMI_SI_3365.raw.vcf,ppmi_wes_645_indiv_vcf_set_01_of_78/PPMI_SI_33...,4045688487,3365.0,True,True
6,ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz,PPMI_SI_3332.raw.vcf,ppmi_wes_645_indiv_vcf_set_01_of_78/PPMI_SI_33...,3485131560,3332.0,True,True
7,ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz,PPMI_SI_3134.raw.vcf,ppmi_wes_645_indiv_vcf_set_01_of_78/PPMI_SI_31...,2116389738,3134.0,True,True
8,ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz,PPMI_SI_3059.raw.vcf,ppmi_wes_645_indiv_vcf_set_01_of_78/PPMI_SI_30...,9275477575,3059.0,True,True


### How many patients does each TAR contain?

In [ ]:
pacientes_por_tar = df_exomas.groupby("tar_file")["patno"].nunique().sort_index()
print(pacientes_por_tar)
print("\nPromedio de pacientes por TAR: {:.1f}".format(pacientes_por_tar.mean()))


tar_file
ppmi_wes_645_indiv_vcf_set_01_of_78.tar.gz    8
ppmi_wes_645_indiv_vcf_set_02_of_78.tar.gz    8
ppmi_wes_645_indiv_vcf_set_03_of_78.tar.gz    9
ppmi_wes_645_indiv_vcf_set_04_of_78.tar.gz    9
ppmi_wes_645_indiv_vcf_set_05_of_78.tar.gz    9
                                             ..
ppmi_wes_645_indiv_vcf_set_74_of_78.tar.gz    5
ppmi_wes_645_indiv_vcf_set_75_of_78.tar.gz    5
ppmi_wes_645_indiv_vcf_set_76_of_78.tar.gz    5
ppmi_wes_645_indiv_vcf_set_77_of_78.tar.gz    5
ppmi_wes_645_indiv_vcf_set_78_of_78.tar.gz    5
Name: patno, Length: 78, dtype: int64

Promedio de pacientes por TAR: 8.3


### Do all file names follow the `PPMI_SI_xxxx.raw.vcf` format?

In [ ]:
no_conformes = df_exomas[~df_exomas["formato_valido"]]
print("Archivos .vcf que NO siguen el patrón PPMI_SI_<numero>.raw.vcf:", len(no_conformes))
if len(no_conformes) > 0:
    display(no_conformes[["tar_file", "nombre_archivo"]].head(20))


Archivos .vcf que NO siguen el patrón PPMI_SI_<numero>.raw.vcf: 0


### How many unique patients have an exome? Are there repeated patients between TARs?

In [ ]:
pacientes_validos = df_exomas[df_exomas["formato_valido"]]

n_pacientes_exoma = pacientes_validos["patno"].nunique()
print("Pacientes únicos con exoma (PATNO extraído del nombre):", n_pacientes_exoma)

conteo_patno = pacientes_validos.groupby("patno")["tar_file"].agg(["nunique", "count"])
conteo_patno.columns = ["n_tars_distintos", "n_apariciones"]

repetidos = conteo_patno[conteo_patno["n_apariciones"] > 1].sort_values("n_apariciones", ascending=False)
print("\nPacientes que aparecen más de una vez (posible duplicado o exoma repetido):", len(repetidos))
if len(repetidos) > 0:
    display(repetidos.head(20))


Pacientes únicos con exoma (PATNO extraído del nombre): 645

Pacientes que aparecen más de una vez (posible duplicado o exoma repetido): 0


## 3. Summary of corrupt TARs

In [ ]:
df_tars_corruptos = pd.DataFrame(tars_corruptos)
if len(df_tars_corruptos) > 0:
    display(df_tars_corruptos)
else:
    print("No se detectaron TAR corruptos ni ilegibles.")


No se detectaron TAR corruptos ni ilegibles.


## 4. Building `exomas_index.csv`

In [ ]:
exomas_index = pacientes_validos[[
    "patno", "tar_file", "nombre_archivo", "tamano_bytes"
]].rename(columns={
    "patno": "PATNO",
    "tar_file": "archivo_tar",
    "nombre_archivo": "archivo_vcf",
    "tamano_bytes": "tamano_bytes"
}).sort_values(["PATNO", "archivo_tar"]).reset_index(drop=True)

print("exomas_index.csv -> filas:", len(exomas_index))
exomas_index.head(10)


exomas_index.csv -> filas: 645


,PATNO,archivo_tar,archivo_vcf,tamano_bytes
0,3000.0,ppmi_wes_645_indiv_vcf_set_26_of_78.tar.gz,PPMI_SI_3000.raw.vcf,4951605430
1,3001.0,ppmi_wes_645_indiv_vcf_set_22_of_78.tar.gz,PPMI_SI_3001.raw.vcf,1815820324
2,3002.0,ppmi_wes_645_indiv_vcf_set_59_of_78.tar.gz,PPMI_SI_3002.raw.vcf,5671439317
3,3003.0,ppmi_wes_645_indiv_vcf_set_18_of_78.tar.gz,PPMI_SI_3003.raw.vcf,3699782627
4,3004.0,ppmi_wes_645_indiv_vcf_set_48_of_78.tar.gz,PPMI_SI_3004.raw.vcf,3384044268
5,3006.0,ppmi_wes_645_indiv_vcf_set_61_of_78.tar.gz,PPMI_SI_3006.raw.vcf,4040252717
6,3008.0,ppmi_wes_645_indiv_vcf_set_68_of_78.tar.gz,PPMI_SI_3008.raw.vcf,4776659661
7,3009.0,ppmi_wes_645_indiv_vcf_set_03_of_78.tar.gz,PPMI_SI_3009.raw.vcf,7917894878
8,3010.0,ppmi_wes_645_indiv_vcf_set_30_of_78.tar.gz,PPMI_SI_3010.raw.vcf,3295778984
9,3011.0,ppmi_wes_645_indiv_vcf_set_17_of_78.tar.gz,PPMI_SI_3011.raw.vcf,2194131828


In [ ]:
ruta_salida = os.path.join(RESULTADOS_DIR, "exomas_index.csv")
exomas_index.to_csv(ruta_salida, index=False)
print("Guardado en:", ruta_salida)


Guardado en: /content/drive/MyDrive/Investigación_Parkinson/Resultados/exomas_index.csv


## Executive Summary

In [ ]:
print("="*60)
print("RESUMEN — Notebook 2: Análisis de exomas")
print("="*60)
print(f"TAR encontrados: {len(tar_files)}")
print(f"TAR con formato de nombre no conforme: {len(tars_no_conformes)}")
print(f"TAR corruptos / ilegibles: {len(tars_corruptos)}")
print(f"Archivos .vcf totales: {len(df_exomas)}")
print(f"Archivos .vcf con nombre no conforme: {len(no_conformes)}")
print(f"Pacientes únicos con exoma: {n_pacientes_exoma}")
print(f"Pacientes con exoma repetido en más de un archivo: {len(repetidos)}")
print(f"Archivo generado: {ruta_salida}")
print("="*60)


RESUMEN — Notebook 2: Análisis de exomas
TAR encontrados: 78
TAR con formato de nombre no conforme: 0
TAR corruptos / ilegibles: 0
Archivos .vcf totales: 645
Archivos .vcf con nombre no conforme: 0
Pacientes únicos con exoma: 645
Pacientes con exoma repetido en más de un archivo: 0
Archivo generado: /content/drive/MyDrive/Investigación_Parkinson/Resultados/exomas_index.csv
